# 13. Download the five full datasets to Google Drive

This notebook only downloads. It does not unzip, resize, label or change anything.

Everything goes to **My Drive › ocdf › datasets**, one folder per dataset:

| Folder | Dataset | From |
|---|---|---|
| `D1_Uganda_cassava_Kaggle` | Cassava Leaf Disease Classification | Kaggle |
| `D3_Ghana_CCMT` | CCMT Crop Pest and Disease Dataset | Mendeley Data, doi 10.17632/bwh3zbpkpv.1 |
| `D4_Tanzania_maize` | Machine Learning Imagery Dataset for Maize, v6.0 | Harvard Dataverse, doi 10.7910/DVN/GDON8Q |
| `D5_Ghana_KaraAgro_maize` | KaraAgro AI Maize Dataset, v4.1 | Harvard Dataverse, doi 10.7910/DVN/CXUMDS |
| `D6_PlantVillage` | PlantVillage | Mendeley Data, doi 10.17632/tywbtsjrjv.1 |

**How to run**
1. Runtime › Change runtime type › **CPU** is enough (no GPU needed).
2. Kaggle needs your token: it should already be in Colab Secrets (key icon) as `KAGGLE_API_TOKEN` from Notebook 01. Make sure "Notebook access" is switched on for it. Do not paste the token into any cell.
3. Run cells 1 and 2, then **read the size printed by cell 3**. The full datasets are large; a free Google Drive has 15 GB. If the total is more than your free space, set the dataset you don't want to `False` in cell 1 and run cell 1 again.
4. Run the remaining cells one at a time. If a cell stops (timeout, disconnect), just run it again: files already downloaded are skipped.

In [ ]:
# 1. Settings: change only this cell
from google.colab import drive
drive.mount('/content/drive')

DEST = '/content/drive/MyDrive/ocdf/datasets'   # where everything goes

DOWNLOAD = {                     # set any to False to skip it
    'D1': True,   # Uganda cassava (Kaggle)
    'D3': True,   # Ghana CCMT (Mendeley)
    'D4': True,   # Tanzania maize (Dataverse)
    'D5': True,   # Ghana KaraAgro maize (Dataverse)
    'D6': True,   # PlantVillage (Mendeley)
}
CCMT_INCLUDE_AUGMENTED = False   # CCMT also has an 'Augmented' copy (~100,000 extra files). The project used Raw only.

import os
os.makedirs(DEST, exist_ok=True)
print('Saving to', DEST)

In [ ]:
# 2. Helpers (nothing to change)
import os, json, time, hashlib, shutil, subprocess, datetime, requests
from concurrent.futures import ThreadPoolExecutor

FOLDER = {'D1': 'D1_Uganda_cassava_Kaggle', 'D3': 'D3_Ghana_CCMT', 'D4': 'D4_Tanzania_maize',
          'D5': 'D5_Ghana_KaraAgro_maize', 'D6': 'D6_PlantVillage'}
DATAVERSE = {'D4': 'doi:10.7910/DVN/GDON8Q', 'D5': 'doi:10.7910/DVN/CXUMDS'}
MENDELEY = {'D3': 'bwh3zbpkpv', 'D6': 'tywbtsjrjv'}
MENDELEY_API = 'https://data.mendeley.com/public-api/datasets'
KNOWN_FOLDERS = {'bwh3zbpkpv': {
  'Raw Data/CCMT Dataset/Cassava/bacterial blight': '5a089bd5-4c26-415c-862d-530233120bd9',
  'Raw Data/CCMT Dataset/Cassava/healthy': '17e79247-49e5-4c6b-b81c-11c97599561d',
  'Raw Data/CCMT Dataset/Cassava/mosaic': 'fbf17af1-ab7c-495b-9c9a-705025a7eadc',
  'Raw Data/CCMT Dataset/Cassava/brown spot': 'd871023f-5dc4-4e10-a4b9-c0b1762ce351',
  'Raw Data/CCMT Dataset/Cassava/green mite': '3d09905e-e4f5-4415-a454-120fd29b6951',
  'Raw Data/CCMT Dataset/Maize/healthy': 'b8b2a5ff-7f79-46c4-8815-ba4cec23cfa1',
  'Raw Data/CCMT Dataset/Maize/streak virus': 'd871ac9d-2d79-425f-9d4a-c5ad0de7800a',
  'Raw Data/CCMT Dataset/Maize/fall armyworm': '7b44bb20-452c-468d-9caf-9149ea7b21e7',
  'Raw Data/CCMT Dataset/Maize/grasshoper': '4da8265a-30d2-49bc-ace2-dbcec2d53aab',
  'Raw Data/CCMT Dataset/Maize/leaf beetle': '4cf175af-5a5a-4191-a3ce-f202b75f0d85',
  'Raw Data/CCMT Dataset/Maize/leaf blight': '08044ee4-b294-4135-b6d4-b7e6ed689618',
  'Raw Data/CCMT Dataset/Maize/leaf spot': '632f07ab-cafd-4752-81ed-8f9914ce64e6',
  'Raw Data/CCMT Dataset/Cashew/healthy': '174b1f48-76a5-42cd-8b39-0f1d2adc2da0',
  'Raw Data/CCMT Dataset/Cashew/anthracnose': 'a1507961-25fd-4ccc-b4ea-dbbc9cbf1aa4',
  'Raw Data/CCMT Dataset/Tomato/healthy': 'aa41f1e8-8303-4e3a-8fb6-61828fd133a7',
  'Raw Data/CCMT Dataset/Tomato/leaf curl': '03ac0d50-4313-47f2-9afb-4bf3166f7fa1'}}

def gb(n): return f'{n / 1e9:.2f} GB'

def md5_file(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def download(url, out, size=None, tries=6):
    """Download url to out. Skips the file if it is already there at the right size. Retries on errors."""
    if os.path.exists(out) and (size is None or os.path.getsize(out) == size):
        return 'skipped'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    for t in range(tries):
        try:
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                with open(out + '.part', 'wb') as f:
                    for blk in r.iter_content(8 << 20): f.write(blk)
            os.replace(out + '.part', out)
            return 'downloaded'
        except Exception as e:
            print(f'  retry {t + 1} for {os.path.basename(out)}: {e}'); time.sleep(10 * (t + 1))
    raise RuntimeError(f'Download failed after {tries} tries: {url}')

def dataverse_files(pid):
    r = requests.get('https://dataverse.harvard.edu/api/datasets/:persistentId/versions/:latest-published/files',
                     params={'persistentId': pid}, timeout=120)
    r.raise_for_status()
    out = []
    for f in r.json()['data']:
        d = f['dataFile']
        rel = os.path.join(f.get('directoryLabel', ''), d['filename'])
        md5 = d.get('md5') or (d.get('checksum', {}).get('value') if d.get('checksum', {}).get('type') == 'MD5' else None)
        out.append({'rel': rel, 'id': d['id'], 'size': d['filesize'], 'md5': md5})
    return out

def mendeley_files(ds, version=1, include_augmented=True):
    """Walk every folder of a public Mendeley Data dataset and return all its files with their folder paths."""
    try:
        r = requests.get(f'{MENDELEY_API}/{ds}/folders/{version}', timeout=120); r.raise_for_status()
        data = r.json()
        if isinstance(data, dict): data = data.get('folders') or data.get('items') or []
        folders = {f['id']: f for f in data if isinstance(f, dict) and 'id' in f}
        if not folders: raise ValueError('empty folder list')
    except Exception as e:
        print(f'  folder list not available ({e}); using the folder ids recorded in Notebooks 01 and 06')
        folders = {fid: {'name': name, 'parent_id': None} for name, fid in KNOWN_FOLDERS.get(ds, {}).items()}
    print(f'  {ds}: {len(folders)} folders found')
    def path(fid):
        parts = []
        while fid in folders:
            parts.append(folders[fid]['name']); fid = folders[fid].get('parent_id')
        return '/'.join(reversed(parts))
    out, seen = [], set()
    for fid in [None] + list(folders):
        start = 0
        while True:
            params = {'version': version, '$start': start, '$limit': 1000}
            if fid: params['folder_id'] = fid
            r = requests.get(f'{MENDELEY_API}/{ds}/files', params=params, timeout=120)
            if r.status_code != 200:
                if fid: print(f'  skipped folder {path(fid)}: HTTP {r.status_code}')
                break
            page = r.json()
            if isinstance(page, dict): page = page.get('files') or page.get('items') or []
            page = [f for f in page if isinstance(f, dict) and 'content_details' in f]
            start += len(page)
            for f in page:
                url = f['content_details']['download_url']
                if url in seen: continue
                seen.add(url)
                rel = os.path.join(path(fid) if fid else '', f['filename'])
                out.append({'rel': rel, 'url': url, 'size': f.get('size') or f['content_details'].get('size'),
                            'sha256': f['content_details'].get('sha256_hash')})
            if len(page) < 1000: break
    if not include_augmented:
        out = [f for f in out if 'augment' not in f['rel'].lower()]
    return out

def save_listing(key, files):
    json.dump({'retrieved_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'files': files},
              open(f'{DEST}/{FOLDER[key]}/_file_list.json', 'w'), indent=1)
print('helpers ready')

In [ ]:
# 3. Preview: what will be downloaded and how big it is (downloads nothing)
LISTS, total = {}, 0
for key in ['D3', 'D4', 'D5', 'D6']:
    if not DOWNLOAD[key]: continue
    if key in DATAVERSE: files = dataverse_files(DATAVERSE[key])
    else: files = mendeley_files(MENDELEY[key], include_augmented=(CCMT_INCLUDE_AUGMENTED if key == 'D3' else True))
    LISTS[key] = files
    size = sum(f['size'] or 0 for f in files); total += size
    print(f'{key} {FOLDER[key]}: {len(files):,} files, {gb(size)}')
if DOWNLOAD['D1']:
    print('D1 Uganda cassava (Kaggle): one zip, size shown when it downloads (several GB)')
print('\nTotal for D3-D6:', gb(total), '+ Kaggle')
try:
    u = shutil.disk_usage('/content/drive/MyDrive'); print('Drive free space reported:', gb(u.free), '(check drive.google.com/settings/storage if unsure)')
except Exception as e:
    print('Could not read Drive free space:', e)

In [ ]:
# 4. D1: Uganda cassava from Kaggle (one zip)
if DOWNLOAD['D1']:
    from google.colab import userdata
    def secret(k):
        try: return userdata.get(k)
        except Exception: return None
    if secret('KAGGLE_API_TOKEN'):
        os.environ['KAGGLE_API_TOKEN'] = secret('KAGGLE_API_TOKEN')
    elif secret('KAGGLE_USERNAME') and secret('KAGGLE_KEY'):
        os.environ['KAGGLE_USERNAME'] = secret('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = secret('KAGGLE_KEY')
    else:
        raise SystemExit('No Kaggle token found in Colab Secrets. Add KAGGLE_API_TOKEN (key icon, left sidebar) and turn on Notebook access.')
    subprocess.run(['pip', 'install', '-q', '-U', 'kaggle'], check=True)
    d = f'{DEST}/{FOLDER["D1"]}'; os.makedirs(d, exist_ok=True)
    zp = f'{d}/cassava-leaf-disease-classification.zip'
    if os.path.exists(zp):
        print('Already downloaded:', zp, gb(os.path.getsize(zp)))
    else:
        r = subprocess.run(['kaggle', 'competitions', 'download', '-c', 'cassava-leaf-disease-classification', '-p', d],
                           capture_output=True, text=True)
        print(r.stdout[-1500:], r.stderr[-1500:])
        if r.returncode != 0:
            raise SystemExit('Kaggle download failed. If it mentions 403 or rules, open the competition page, accept the rules, and run this cell again.')
        print('Saved:', zp, gb(os.path.getsize(zp)))
else:
    print('D1 skipped')

In [ ]:
# 5. D4 Tanzania maize and D5 KaraAgro maize from Harvard Dataverse (every file, MD5 checked)
for key in ['D4', 'D5']:
    if not DOWNLOAD[key]: print(key, 'skipped'); continue
    files = LISTS.get(key) or dataverse_files(DATAVERSE[key])
    d = f'{DEST}/{FOLDER[key]}'; os.makedirs(d, exist_ok=True); save_listing(key, files)
    for i, f in enumerate(files, 1):
        out = f'{d}/{f["rel"]}'
        status = download(f'https://dataverse.harvard.edu/api/access/datafile/{f["id"]}', out, f['size'])
        ok = (f['md5'] is None) or (status == 'skipped') or (md5_file(out) == f['md5'])
        print(f'{key} [{i}/{len(files)}] {f["rel"]} ({gb(f["size"])}): {status}{"" if ok else "  MD5 MISMATCH - delete this file and run the cell again"}')
    print(key, 'done\n')

In [ ]:
# 6. D3 Ghana CCMT and D6 PlantVillage from Mendeley Data (every file, SHA-256 checked)
for key in ['D3', 'D6']:
    if not DOWNLOAD[key]: print(key, 'skipped'); continue
    files = LISTS.get(key) or mendeley_files(MENDELEY[key], include_augmented=(CCMT_INCLUDE_AUGMENTED if key == 'D3' else True))
    d = f'{DEST}/{FOLDER[key]}'; os.makedirs(d, exist_ok=True); save_listing(key, files)
    def fetch(f):
        out = f'{d}/{f["rel"]}'
        status = download(f['url'], out, f['size'])
        if status == 'downloaded' and f.get('sha256') and sha256_file(out) != f['sha256']:
            os.remove(out); return 'mismatch'
        return status
    t0 = time.time()
    workers = 16 if len(files) > 50 else 2
    with ThreadPoolExecutor(workers) as ex: results = list(ex.map(fetch, files))
    from collections import Counter
    print(key, dict(Counter(results)), f'in {time.time() - t0:.0f}s')
    if 'mismatch' in results: print('  Some files did not match their checksum and were removed. Run this cell again to fetch them.')

In [ ]:
# 7. Summary: what is now in the dataset folder
for key, name in FOLDER.items():
    p = f'{DEST}/{name}'
    if not os.path.isdir(p): print(f'{key} {name}: not downloaded'); continue
    n = size = 0
    for root, _, fs in os.walk(p):
        for f in fs:
            if f.endswith('.part') or f == '_file_list.json': continue
            n += 1; size += os.path.getsize(os.path.join(root, f))
    print(f'{key} {name}: {n:,} files, {gb(size)}')
print('\nDone. Each folder has a _file_list.json recording exactly what was downloaded and when.')